![Portada](recursos/img/portada_eda_matrix.png)

## 📖 Descripción del EDA

En 1999, el mundo se encontró con una pregunta incómoda: **¿y si todo lo que percibes como real no lo es?**

En _The Matrix_, el protagonista descubre que **la humanidad vive atrapada en una simulación digital** creada por máquinas, mientras sus cuerpos son utilizados como fuente de energía. 

La película consiguió combinar filosofía, acción y efectos visuales nunca vistos hasta entonces y con sus sagas dividió al público entre quienes la vivieron como puro entretenimiento y quienes vieron en ella algo mucho más profundo.

Este EDA (Análisis Exploratorio de Datos) nace precisamente de esa batalla de audiencia. Y es a través de los datos que busco explorar el impacto real de la saga. Por eso, me pregunto si realmente revolucionó el cine de ciencia ficción, si su temática trascendió la pantalla hasta llegar a la comunidad científica y si las secuelas fueron realmente peores o simplemente más difíciles de comprender.

## 💊 Hipótesis

#### 1.- **_The Matrix_ revolucionó el género de la ciencia ficción** 

La película estableció un nuevo estándar de inversión en películas de ciencia ficción. A partir de entonces, las películas del género optaron por impulsar los efectos digitales, con lo que los presupuestos y los especialistas en VFX aumentaron en número por el uso de nuevas tecnologías para dejar con la boca abierta a los espectadores.

#### 2.- **La realidad como simulación: ciencia ficción vs papers**

Existe un incremento significativo en la producción académica sobre la [_Teoría de la Simulación_](https://es.wikipedia.org/wiki/Hip%C3%B3tesis_de_simulaci%C3%B3n) en el campo de la Astrofísica y la Cosmología a raíz del estreno de _The Matrix_, consolidando a la saga como un referente más allá de la ficción.

#### 3.- **El declive de la saga: ¿motivada por el descenso de calidad?**

Se propone que con una mayor narrativa y más términos complejos, menor es la conexión emocional y la valoración del público general. En realidad, quizás las películas no eran peores, sino que no llegaron a ser comprendidas.

## 🔌 Datos

Dado que la temática de las hipótesis es variada, también lo serán las distintas fuentes de datos que tengo que utilizar para poder contrastarlas. **A continuación muestro cada hipótesis y las fuentes** que serán utilizadas:

#### Fuentes Hipótesis 1 (__The Matrix_ revolucionó el género de la ciencia ficción_)
Utilizo datasets de métricas de cine, obtenidos de Kaggle. En concreto, uso [IMDb Dataset](https://www.kaggle.com/datasets/ashirwadsangwan/imdb-dataset) y [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata).

Trabajando el dataset de [Tmbd](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata) puedo estudiar los **presupuestos** de las películas por año:

In [ ]:
import pandas as pd

ruta = "src/datos/brutos/hipotesis_1/tmdb_5000_movies.csv"

peliculas = pd.read_csv(ruta)

peliculas[["original_title", "release_date", "budget", "genres"]]

,original_title,release_date,budget,genres
0,Avatar,2009-12-10,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
1,Pirates of the Caribbean: At World's End,2007-05-19,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""..."
2,Spectre,2015-10-26,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
3,The Dark Knight Rises,2012-07-16,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam..."
4,John Carter,2012-03-07,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
...,...,...,...,...
4798,El Mariachi,1992-09-04,220000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam..."
4799,Newlyweds,2011-12-26,9000,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""..."
4800,"Signed, Sealed, Delivered",2013-10-13,0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 18, ""nam..."
4801,Shanghai Calling,2012-05-03,0,[]


Además, también utilizaré otro dataset del mismo conjunto para poder encontrar la **cantidad de especialistas en efectos especiales** que trabajan en ciencia ficción en las últimas décadas:

In [72]:
import pandas as pd
import ast

creditos = pd.read_csv("src/datos/brutos/hipotesis_1/tmdb_5000_credits.csv")

# Convierto el string en diccionario para trabajar con ella
creditos["crew"] = creditos["crew"].apply(ast.literal_eval)

# Extraigo solo gente de efectos especiales (también llamados visuales) y quedarnos con los campos útiles
def extraer_vfx(crew):
    return [
        {"nombre": p["name"], "rol": p["job"]}
        for p in crew
        if p["department"] == "Visual Effects"
    ]

creditos["vfx"] = creditos["crew"].apply(extraer_vfx)

# Convertir cada persona en una fila
df_vfx = creditos[["title", "vfx"]].explode("vfx").dropna(subset=["vfx"])
df_vfx = df_vfx.join(pd.json_normalize(df_vfx.pop("vfx")))

# Muestro a las 10 primeras personas
df_vfx.head(10).reset_index(drop=True)

,title,nombre,rol
0,Avatar,Jill Brooks,Visual Effects Producer
1,Avatar,Richard Martin,Visual Effects Supervisor
2,Avatar,Steven Quale,Visual Effects Supervisor
3,Avatar,Jonathan Rothbart,Visual Effects Supervisor
4,Avatar,Alain Lalanne,Visual Effects Producer
5,Avatar,Lucas Salton,Visual Effects Supervisor
6,Avatar,Stephen Rosenbaum,Visual Effects Supervisor
7,Avatar,Jonathan Fawkner,Visual Effects Supervisor
8,Avatar,Chris Del Conte,Visual Effects Producer
9,Avatar,R. Christopher White,Visual Effects Supervisor


#### Fuentes Hipótesis 2 (_La realidad como simulación: ciencia ficción vs papers_)
Para contrastar la hipótesis uso las **librerías arXiv y scholarly**. Además, también miraré si me resulta de utilidad la API de [OpenAlex](https://openalex.org/). Aquí debajo muestro un ejemplo del uso de la librería de arXiv para poder encontrar papers científicos que remitan a conceptos de simulación.

In [71]:
import arxiv

cliente = arxiv.Client()

busqueda = arxiv.Search(
    query="simulation theory reality",
    max_results=5,
    sort_by=arxiv.SortCriterion.Relevance
)

resultados = []
for paper in cliente.results(busqueda):
    resultados.append({
        "título": paper.title,
        "autores": ", ".join(a.name for a in paper.authors[:2]) + " et al.",
        "año": paper.published.year,
        "categoría": paper.primary_category
    })

df_papers = pd.DataFrame(resultados)
df_papers

,título,autores,año,categoría
0,Fitted avatars: automatic skeleton adjustment ...,"Jose Luis Ponton, Víctor Ceballos et al.",2023,cs.HC
1,Interactive Multi-User 3D Visual Analytics in ...,"Wanze Xie, Yining Liang et al.",2020,cs.HC
2,Real-Time Detection of Simulator Sickness in V...,"Jialin Wang, Hai-Ning Liang et al.",2020,cs.HC
3,Variational Approach to Quantum Field Theory: ...,Jae Hyung Yee et al.,1997,hep-th
4,Reality Distortion Room: A Study of User Locom...,"You-Jin Kim, Andrew D. Wilson et al.",2025,cs.HC


#### Fuentes Hipótesis 3 (_El declive de la saga: ¿motivada por el descenso de calidad?_)
Para acabar he eligo unos datasets muy curiosos, ya que cada uno de ellos contiene las transcripciones de las películas, que me servirán para estudiar si los guiones eran peores o, simplemente, se fueron complicando. Los datasets están agrupados en Kaggle bajo el nombre [The Matrix](https://www.kaggle.com/datasets/nixongeno/the-matrix-movie-transcripts). Para el procesamiento de las transcripciones se utilizarán dos librerías especializadas en procesamiento de lenguaje natural: NLTK y SpaCy.

In [54]:
import pandas as pd

# 1. Ruta directa al archivo específico
ruta = "src/datos/brutos/hipotesis_3/the_matrix.csv"

# 2. Cargar el archivo con el encoding que ya sabemos que funciona
df_guion = pd.read_csv(ruta, encoding='latin1')

# 4. Mostrar el resultado
print("Muestra del guion de 'The Matrix':")
display(df_guion.sample(5, random_state=42)) # muestra solo una parte aleatoria del guion

Muestra del guion de 'The Matrix':


,Character_Name,Character_dialogue
249,Morpheus,Your mind makes it real.
399,Tank,Oh my God.
174,Morpheus,"Welcome to the real world We've done it, Trin..."
280,Trinity,There old service and waste systems.
110,Neo,"Yeah. Wow, that sound like a really good deal..."


## 📂 Estructura del [Repositorio](https://github.com/RobertoCantero82/EDA_matrix_decodificada)

#### CARPETAS:

* **presentacion/:** se incluye el archivo de soporte para la exposición del EDA (10 minutos).

* **recursos/:** carpeta con material multimedia (img, audio, video), que servirá de apoyo en la presentación.

* **src/:** carpeta que contiene la parte fundamental del proyecto.  

    * **codigo/:** módulos de Python y archivos necesarios para que el EDA sea funcional.

    * **datos/:** datasets originales y procesados.

    * **notebooks/:** cuadernos de Jupyter con pruebas realizadas para plantear hipótesis y su resolución.

    * **memoria.ipynb:** cuaderno principal con el desarrollo completo, limpiezas, visualizaciones y conclusiones.

#### ARCHIVOS: 

* **presentacion_EDA**: explicación inicial del proyecto, donde se incluye el tema elegido, una breve explicación de la película y la saga, las hipótesis planteadas y cómo serán los primeros pasos para poder obtener conclusiones profesionales.

* **README.md:** archivo que será el punto de partida de quien entra al repositorio de GitHub para conocer qué es lo que se encontrará al acceder al EDA. 

* **.gitignore:** utilizo este archivo para evitar que se suban a GitHub los datasets brutos y los archivos y carpetas temporales.